# 02 Interaction features and hyperparameter tuning: Two-stage, Ridge trend + random forest

Part of `03b_training_pipeline_model/two_stage_rf_ridge`; reads `feature_config.py` (written by notebook 01) and writes `model_config.py`.

The model is fit on the train split (2021-01 to 2024-09) and scored on the validation split (2024-10 to 2025-03) for three training windows: the **full
history** (from 2021-01), **2023 onward** (from 2023-01) and **after May 2023** (from 2023-06 only).

- Trend: a **Ridge** on the selected temporal features forecasts the monthly level `alpha_m` that was decomposed from the static features (Ridge of log
  rent on one dummy per year-month plus the static features). The temporal features are standardised on the training months.
- Static part: a random forest on log rent minus the trend, on the static features, optionally with **interaction features** (every numeric static
  feature times every selected temporal feature of the flat's month). Interactions on or off is tuned.
- The forest fits do not depend on the trend penalty, so each forest is combined with every Ridge penalty without refitting.

The setup is chosen per window by **validation log RMSE** (log R2, dollar R2 and dollar RMSE are reported alongside).

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

SHARED_DIR = '/Users/keltonn/Desktop/Projects/housing_price_regressor/03b_training_pipeline_model'
FOLDER = f'{SHARED_DIR}/two_stage_rf_ridge'
sys.path.insert(0, SHARED_DIR)      # pipeline_utils.py, shared by every model folder
sys.path.insert(0, FOLDER)          # this folder's feature_config.py and model_config.py
import pprint
from sklearn.linear_model import Lasso, Ridge, lasso_path
from sklearn.preprocessing import StandardScaler
import feature_config
import pipeline_utils as pu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

TREND_ALPHAS = [1e-2, 1e-1, 1, 3, 10, 30, 100, 300, 1e3, 1e4]         # trend Ridge, on features standardised over ~45 months
RF_GRID = [
    dict(n_estimators=200, max_depth=14, min_samples_leaf=5, max_features=64, max_samples=0.6),
    dict(n_estimators=200, max_depth=14, min_samples_leaf=10, max_features=128, max_samples=0.6),
    dict(n_estimators=200, max_depth=24, min_samples_leaf=3, max_features=64, max_samples=0.6),
    dict(n_estimators=200, max_depth=24, min_samples_leaf=10, max_features=128, max_samples=0.6),
    dict(n_estimators=200, max_depth=None, min_samples_leaf=5, max_features=96, max_samples=0.6),
    dict(n_estimators=200, max_depth=None, min_samples_leaf=20, max_features=96, max_samples=0.6),
]
FAMILY = 'two_stage_rf_ridge'

## 1. Data

In [2]:
frames, numeric, onehot = pu.load_static()
temporal = pu.load_temporal()
n_num = len(numeric)
tr, va = frames['train'], frames['validation']
X_tr, y_tr, m_tr = pu.design(tr, numeric, onehot), pu.log_rent(tr), tr['month'].to_numpy()
X_va, y_va, m_va = pu.design(va, numeric, onehot), pu.log_rent(va), va['month'].to_numpy()
val_months = np.arange(pu.VALIDATION_FIRST, pu.VALIDATION_LAST + 1)
print(f'train {X_tr.shape}, validation {X_va.shape}')

train (132398, 237), validation (16961, 237)


## 2. Tune the Ridge-trend + random forest model

For each window: decompose the trend on the window's training months, forecast the validation months with the Ridge trend model for every penalty, fit
the random forest for every setting (with and without interaction features) on the residual, and score every combination on the validation rows. Trend
and static predictions add in log space.

In [3]:
def evaluate(window):
    start = pu.WINDOW_START[window]
    months_fit = np.arange(start, pu.TRAIN_LAST + 1)
    s1_months = months_fit[months_fit >= 1]                           # 2021-01 has no month-over-month return
    alpha_m = pu.decompose_trend(X_tr, y_tr, m_tr, months_fit)        # trend decomposed from the static features
    selected = feature_config.SELECTED_TEMPORAL_FEATURES[f'{window}/train']
    Z = pu.zscore(temporal, selected, s1_months)

    trends = {}                                                       # penalty -> (validation-month forecast, features used)
    for a in TREND_ALPHAS:
        model = Ridge(alpha=a).fit(Z.loc[s1_months], alpha_m.loc[s1_months])
        n_used = len(selected)
        trends[a] = (pd.Series(model.predict(Z.loc[val_months]), index=val_months), n_used)

    rows = np.flatnonzero(m_tr >= start)                              # the static model trains on the rows of the window only
    d_tr = y_tr[rows] - alpha_m.loc[m_tr[rows]].values
    out = []
    for use_interactions in (False, True):
        Xa = pu.static_design(X_tr[rows], n_num, Z, m_tr[rows], use_interactions)
        Xv = pu.static_design(X_va, n_num, Z, m_va, use_interactions)
        for params in RF_GRID:
            d_val = pu.make_static_model(params).fit(Xa, d_tr).predict(Xv)
            for a, (forecast, n_used) in trends.items():
                out.append({'window': window, 'family': FAMILY, 'interactions': use_interactions, 'static_params': params, 'alpha': a,
                            'features_used': n_used, **pu.score(y_va, forecast.loc[m_va].values + d_val)})
        print(f'{window}: interactions={use_interactions} done ({Xa.shape[1]} static columns)')
    return out


results = pd.DataFrame([row for window in pu.WINDOW_START for row in evaluate(window)])
print(len(results), 'combinations scored')

full_history: interactions=False done (237 static columns)


full_history: interactions=True done (549 static columns)


from_2023: interactions=False done (237 static columns)


from_2023: interactions=True done (627 static columns)


from_2023_06: interactions=False done (237 static columns)


from_2023_06: interactions=True done (549 static columns)
360 combinations scored


## 3. Best setup per window

In [4]:
best = results.loc[results.groupby('window')['log_rmse'].idxmin()].set_index('window')
view = best[['log_r2', 'dollar_r2', 'log_rmse', 'dollar_rmse']].copy()
view['settings'] = best.apply(lambda r: f"interactions={r['interactions']}, trend alpha={r['alpha']}, {int(r['features_used'])} trend features, " +
                              ', '.join(f'{k}={v}' for k, v in r['static_params'].items()), axis=1)
display(view.sort_values('log_rmse').round(4))

,log_r2,dollar_r2,log_rmse,dollar_rmse,settings
window,,,,,
full_history,0.4167,0.4855,0.1739,460.2335,"interactions=False, trend alpha=0.01, 8 trend features, n_estimators=200, max_depth=24, min_samples_leaf=10, max_fea..."
from_2023,0.4035,0.4715,0.1759,466.4684,"interactions=False, trend alpha=10.0, 10 trend features, n_estimators=200, max_depth=14, min_samples_leaf=10, max_fe..."
from_2023_06,0.3962,0.4764,0.1769,464.2900,"interactions=False, trend alpha=0.01, 8 trend features, n_estimators=200, max_depth=None, min_samples_leaf=20, max_f..."


In [5]:
# what the interaction features and the trend penalty do, holding the rest at its best value
display(results.groupby(['window', 'interactions'])['log_rmse'].min().unstack('interactions')
        .rename(columns={False: 'best log RMSE without interactions', True: 'best log RMSE with interactions'}).round(4))
display(results.groupby(['alpha', 'window'])['log_rmse'].min().unstack('window').round(4).rename_axis('best log RMSE by Ridge trend penalty'))

interactions,best log RMSE without interactions,best log RMSE with interactions
window,,
from_2023,0.1759,0.1833
from_2023_06,0.1769,0.1795
full_history,0.1739,0.1793


window,from_2023,from_2023_06,full_history
best log RMSE by Ridge trend penalty,,,
0.01,0.1826,0.1769,0.1739
0.10,0.1809,0.1769,0.1744
1.00,0.1773,0.1770,0.1770
3.00,0.1763,0.1771,0.1784
10.00,0.1759,0.1773,0.1803
30.00,0.1761,0.1776,0.1852
100.00,0.1767,0.1780,0.1968
300.00,0.1773,0.1783,0.2128
1000.00,0.1776,0.1784,0.2294


## 4. Write `model_config.py`

In [6]:
def plain(params):
    return {k: (float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) and not isinstance(v, bool) else v)
            for k, v in params.items()}


def metrics(row):
    return {m: round(float(row[m]), 5) for m in ('log_r2', 'dollar_r2', 'log_rmse', 'dollar_rmse')}


setups = {}
for window in pu.WINDOW_START:
    r = best.loc[window]
    setups[f'{window}/{FAMILY}'] = {'interactions': bool(r['interactions']), 'ridge_alpha': float(r['alpha']), 'static_params': plain(r['static_params']), 'validation': metrics(r)}
text = ('"""Best setup per training window for this model folder, chosen on the validation set (lowest log RMSE) by 02_tune_model.ipynb.\n'
        'Rerun that notebook to update it."""\n\nBEST_SETUPS = ' + pprint.pformat(setups, width=140, sort_dicts=False) + '\n')
with open(f'{FOLDER}/model_config.py', 'w') as f:
    f.write(text)
print(text)

"""Best setup per training window for this model folder, chosen on the validation set (lowest log RMSE) by 02_tune_model.ipynb.
Rerun that notebook to update it."""

BEST_SETUPS = {'full_history/two_stage_rf_ridge': {'interactions': False,
                                     'ridge_alpha': 0.01,
                                     'static_params': {'n_estimators': 200,
                                                       'max_depth': 24,
                                                       'min_samples_leaf': 10,
                                                       'max_features': 128,
                                                       'max_samples': 0.6},
                                     'validation': {'log_r2': 0.41672,
                                                    'dollar_r2': 0.48553,
                                                    'log_rmse': 0.17392,
                                                    'dollar_rmse': 460.23347}},
 'from_2023/two_stage_rf_